# Chapter 37: the financial machine-learning problem

Fresh constructed calculations contrast accuracy with net return, check the absolute-return-weighted accuracy identity, simulate overlapping labels and selected null Sharpe, then examine a regime shift. Signals form at the prior close, positions enter at the next open, and the return spans that open to the following open. No market evidence or Project 6 validation is produced.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch37_ml_philosophy_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import importlib.util
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def load_chapter(name):
    spec = importlib.util.spec_from_file_location(name, ROOT / "code" / f"{name}.py")
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

ch37 = load_chapter('ch37_ml_philosophy')

## 1. Ten-day accuracy and net return

Cost is 0.1% per unit turnover, initial position is zero, and the final position is not liquidated. Compare A's lower accuracy with B's higher accuracy using both approximate net sums and exact compounded wealth. The path has 11 wealth points for ten returns.

In [ ]:
case = ch37.ten_day_case()
rows = []
for name in ("A_lower_accuracy", "B_high_accuracy", "buy_and_hold", "zero_position"):
    r = case[name]
    rows.append({"model": name, "accuracy": r["accuracy"], "sum_gross": r["sum_gross"], "turnover": r["total_turnover"],
                 "cost": r["sum_cost"], "approx_net": r["sum_net_approx"], "exact_net": r["compound_net_return_exact"]})
table = pd.DataFrame(rows).set_index("model")
display(table.round(6))
assert abs(case["A_lower_accuracy"]["compound_net_return_exact"] - 0.056879) < 1e-6
assert abs(case["B_high_accuracy"]["compound_net_return_exact"] + 0.039980) < 1e-6
assert len(case["A_lower_accuracy"]["wealth_points"]) == 11      # 10 returns -> 11 wealth points

## 2. A weighted-accuracy identity

For positions restricted to ±1, $\sum p_tr_t=(\sum|r_t|)(2\mathrm{Acc}_{|r|}-1)$. Confidence-scaled position sizes invalidate this particular identity.

In [ ]:
for name in ("A_lower_accuracy", "B_high_accuracy"):
    ident = case[name]["identity"]
    print(name, {k: round(v, 6) for k, v in ident.items()})
    assert abs(ident["lhs_sum_pos_r"] - ident["rhs_abs_weighted"]) < 1e-12

## 3. Overlapping labels

Simulate 100,000 daily increments with horizon 5. The theoretical autocorrelation is $\rho_k=(h-k)/h$ over overlapping lags; the mean's variance inflation approaches h under the stated independent-increment mechanism.

In [ ]:
ov = ch37.overlap_autocorrelation(h=5, n_days=100_000, seed=3703)
print("sample", np.round(ov["sample_rho"], 3), "theory", ov["theory_rho"])
assert max(abs(a - b) for a, b in zip(ov["sample_rho"], ov["theory_rho"])) < 0.02

## 4. Selecting the best null Sharpe

Freshly simulate 200 repetitions of 504 days for 10, 100 and 1,000 strategies. Compare selected in-sample Sharpe with the expected-maximum formula. Population holdout expectation is zero, but a finite Monte Carlo average need not be zero or statistically indistinguishable from it. The |mean|<0.25 assertion is an engineering tolerance, not a significance test. This Notebook omits the CLI's N=1 case before consuming the same random stream, so subsequent values need not equal the CLI's saved results despite the shared seed.

In [ ]:
mt = ch37.max_null_sharpe((10, 100, 1000), n_days=504, n_sims=200, seed=3702)
print(pd.DataFrame([{k: r[k] for k in ("n_strategies", "mean_best_in_sample", "formula_exact_quadrature",
                                           "mean_holdout_of_selected")} for r in mt["rows"]]).round(3))
for r in mt["rows"]:
    assert abs(r["mean_best_in_sample"] - r["formula_exact_quadrature"]) < 0.15 and abs(r["mean_holdout_of_selected"]) < 0.25

## 5. Regime shift and figure

Fit on 1,000 observations from regime 1, then evaluate regime 2. Reversing the slope makes R² relative to zero negative in this construction. Save one new ten-day wealth comparison under RUN_DIR; the broader simulations above can cost substantially more than hand checks.

In [ ]:
rg = ch37.regime_shift(1000, 0.1, 3701)
print("train beta", round(rg["train_beta"], 4), "R2_OS vs zero", round(rg["oos_r2_vs_zero_regime2"], 4))
assert rg["oos_r2_vs_zero_regime2"] < 0
w = case["A_lower_accuracy"]["wealth_points"]; v = case["B_high_accuracy"]["wealth_points"]
plt.plot(w, marker="o", label="A (70%)"); plt.plot(v, marker="o", label="B (80%)"); plt.axhline(1, color="grey", lw=0.6)
plt.xlabel("wealth point"); plt.ylabel("wealth (start 1.0)"); plt.legend(); plt.title("CONSTRUCTED ten-day case"); plt.savefig(RUN_DIR / "ch37_ml_philosophy_cell11.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()